# CSS 551 · HW8 · Diffusion and learned scenes (Python track)

Fill in every function marked `TODO`, run all cells, and submit this notebook **run**, with
the check cell's output visible. The functions and numbers are the JavaScript track's (see
the homework page). NumPy is allowed; no diffusion, rendering or splatting libraries. Keep
`data.json` and `expected.json` next to this notebook.

In [ ]:
import json, math
import numpy as np

DATA = json.load(open("data.json"))
EXP = json.load(open("expected.json"))
INPUTS = EXP["inputs"]

In [ ]:
def alpha_bar(t):
    """GIVEN (not graded): the course's cosine schedule, t in [0, 1]."""
    S = 0.008
    f = lambda u: math.cos((u + S) / (1 + S) * math.pi / 2) ** 2
    u = min(1.0, max(0.0, t))
    return min(1.0, max(0.0, f(u) / f(0)))

In [ ]:
def denoise(x, t, data):
    """The exact denoiser: sum_i w_i data_i, softmax weights w_i proportional to
    exp(-|x - sqrt(abar) data_i|^2 / (2 (1 - abar))), abar = alpha_bar(t), 1 - abar floored
    at 1e-6. Subtract the largest logit before exp. x is (2,), data is (N, 2)."""
    # TODO
    return np.zeros(2)

In [ ]:
def ddim_step(xt, x0hat, t_from, t_to):
    """eps = (x_t - sqrt(abar1) x0hat) / sqrt(1 - abar1); x_next = sqrt(abar2) x0hat + sqrt(1 - abar2) eps."""
    # TODO
    return np.asarray(xt, float)

In [ ]:
def timeline(steps, t_max=0.999):
    """GIVEN: the sampler's times, t_max down to 0 in `steps` equal steps."""
    return [0.0 if k == steps else t_max * (1 - k / steps) for k in range(steps + 1)]

In [ ]:
def sample_ddim(starts, data, steps):
    """Every start through `steps` DDIM steps along timeline(steps); returns the final points."""
    # TODO: for k in range(steps), step each point from ts[k] to ts[k + 1].
    return np.asarray(starts, float)

In [ ]:
def render_ray(sigmas, colors, delta):
    """alpha_i = 1 - exp(-sigma_i delta); weight T_i alpha_i with T_i the product of
    (1 - alpha_j), j < i. Returns (C, T): the summed color and the transmittance left."""
    # TODO
    return np.zeros(3), 1.0

In [ ]:
def project_splat(s, rot_z_deg, t, f):
    """Sigma = M M^T, M = R_z(rot_z_deg) diag(s); J = [[f/z, 0, -f x/z^2], [0, f/z, -f y/z^2]]
    at the center t = (x, y, z). Returns (cov, radii, angle_deg, center): cov = [xx, xy, yy] of
    J Sigma J^T, radii = sqrt of its eigenvalues (larger first), angle = 0.5 atan2(2 xy, xx - yy)
    in degrees, center = (f x / z, f y / z)."""
    # TODO
    return [0.0, 0.0, 0.0], [0.0, 0.0], 0.0, [0.0, 0.0]

In [ ]:
def composite_splats(splats, u):
    """Front to back: alpha_k = a_k exp(-(u - mu_k)^2 / (2 sd_k^2)); C = sum_k T_k alpha_k c_k.
    Returns (C, T)."""
    # TODO
    return np.zeros(3), 1.0

## Checks

Run this last. It prints each check with your value and the expected one, and the score on the published inputs. Grading also runs your functions on inputs that are not published.

In [ ]:
def flat(v):
    if isinstance(v, dict):
        raise TypeError("a check returned a dict; return the values in the order the check asks for")
    if isinstance(v, (list, tuple, np.ndarray)):
        out = []
        for x in v:
            out.extend(flat(x))
        return out
    if isinstance(v, (bool, np.bool_)):
        return [bool(v)]
    return [float(v)]

def same(got, want, tol):
    if want is None or got is None:
        return got is None and want is None
    g, w = flat(got), flat(want)
    if len(g) != len(w):
        return False
    return all(abs(a - b) <= tol and not math.isnan(a) for a, b in zip(g, w))

def report(runs):
    score, lines = 0, []
    for c in EXP["checks"]:
        try:
            got, err = runs[c["id"]](), None
        except Exception as e:  # a stub or a bug: the check fails, the rest still run
            got, err = None, repr(e)
        ok = err is None and same(got, c["expected"], c["tol"])
        score += c["points"] if ok else 0
        shown = err if err else np.round(np.array(flat(got), dtype=float), 4).tolist() if got is not None else None
        lines.append(f'{"PASS" if ok else "FAIL"}  {c["id"]:<8} {c["points"]:>3} pts  {c["label"]}\n      yours {shown}\n      expected {c["expected"]}')
    print("\n".join(lines))
    print(f"\nscore on the published inputs: {score} / 100")
    if score == 100:
        print("ALL CHECKS PASS")


spiral = np.asarray(DATA["spiral"], float)

def mean_nearest(points):
    P = np.asarray(points, float)
    d = np.sqrt(((P[:, None, :] - spiral[None, :, :]) ** 2).sum(axis=2))
    return float(d.min(axis=1).mean())

st, v, sp, row = INPUTS["step"], INPUTS["volume"], INPUTS["splat"], INPUTS["row"]
report({
    "x0hat": lambda: denoise(st["xt"], st["tFrom"], spiral),
    "ddim": lambda: ddim_step(st["xt"], denoise(st["xt"], st["tFrom"], spiral), st["tFrom"], st["tTo"]),
    "sampler": lambda: mean_nearest(sample_ddim(DATA["starts"][INPUTS["sampler"].get("offset", 0):INPUTS["sampler"].get("offset", 0) + INPUTS["sampler"]["count"]], spiral, INPUTS["sampler"]["steps"])),
    "rayC": lambda: render_ray(v["sigmas"], v["colors"], v["delta"])[0],
    "rayT": lambda: render_ray(v["sigmas"], v["colors"], v["delta"])[1],
    "cov": lambda: project_splat(sp["s"], sp["rotZ"], sp["t"], sp["f"])[0],
    "radii": lambda: list(project_splat(sp["s"], sp["rotZ"], sp["t"], sp["f"])[1:]),
    "splatC": lambda: list(composite_splats(row["splats"], row["u"])),
})